In [107]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [108]:
data9 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/ENEMDU2/2020/BDD_ENEMDU_2020_09_SPSS/enemdu_personas_2020_09.sav", convert_categoricals=False) # para bases de stata
data12 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/ENEMDU2/2020/BBDD_PUBLICACION_ DIC 20_SPSS/BBDD_PUBLICACION_ DIC 20_SPSS/enemdu_persona_2020_12.sav", convert_categoricals=False) # para bases de stata

#data9 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2020\BDD_ENEMDU_2020_09_SPSS\enemdu_personas_2020_09.sav", convert_categoricals=False) # para bases de stata
#data12 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2020\BBDD_PUBLICACION_ DIC 20_SPSS\BBDD_PUBLICACION_ DIC 20_SPSS\enemdu_persona_2020_12.sav", convert_categoricals=False) # para bases de stata

In [109]:
data9.columns = [x.lower() for x in data9.columns]
data12.columns = [x.lower() for x in data12.columns]

## Revisar los datos

| septiembre | diciembre |
|-----------|-----------|
| area  | area  |
| upm | ciudad  |
|  | panelm  |
| vivienda  | vivienda  |
| hogar  | hogar  |
| p02  | p02  |
| p03  | p03  |
| p66  | p66  |
| fexp  | fexp  |
| p20  | p20  |
| id_hogar  | id_hogar  |

En esta encuesta tenemos separadas cuatro diferentes bases para cada trimestre, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

En ests encuesta de 2018, el primer trimestre no tiene la variable de ciudad, zona y sector en este caso utilizaremos como proxy los primeros dígitos de la UPM que por lo general incluye geocódigos ya que tampoco está disponible variable p15ab

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, estas variables las usamos antes para identificar la condición de trabajo para diferentes meses en encuestas anuales o incompletas donde asumíamos que mantenía el mismo salario si estaba ocupado en ese mes, sin mbargo estas variables tenían el problema de no corresponder de forma exacta con el año o mes de la encuesta. Ahora sin embargo podemos cambiar las suposiciones y solamente asumir que si la variable 'trabajando' que pregunta si el individuo trabajó la semana pasada se cumple vamos a asumir que trabajo durante todo el trimestre, de esta manera podemos mejorar las suposiciones de ocupación mensual, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [110]:
columnas = pd.Index(['area', 'ciudad', 'upm', 'panelm',
            'vivienda', 'hogar', 'p66',
            'fexp', 'p02', 'p03', 'p20', 'id_hogar'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [111]:
data9 = data9[columnas.intersection(data9.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [112]:
data12['p66'].value_counts().sort_index(ascending=False)

p66
999999.0     79
5000.0        2
4000.0        3
3500.0        1
3440.0        1
           ... 
20.0          5
17.0          1
12.0          2
10.0          4
0.0         166
Name: count, Length: 401, dtype: int64

In [113]:
data9['p66'] = pd.to_numeric(data9['p66'], errors='coerce')
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x > 5000 else x)
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x < 0 else x)

data12['p66'] = pd.to_numeric(data12['p66'], errors='coerce')
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x > 5000 else x)
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x < 0 else x)

In [114]:
data9['ingr'] = data9['p66']
data12['ingr'] = data12['p66']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [115]:
data9['ingr_t3'] = data9.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [116]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2020]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [117]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [118]:
data9['ciudad'] = data9['upm'].apply(lambda x: x[:5])

In [119]:
print(data9['ciudad'][0])
fac9 = data9['ciudad'].apply(lambda x: len(str(x))).min()
print(fac9)

print(data12['ciudad'][0])
fac12 = data12['ciudad'].apply(lambda x: len(str(x))).min()
print(fac12)

01015
5
200250.0
7


In [120]:
# Corregimos los códigos para usarlos cómo texto
data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == fac9-1 else x)
data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:4])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == fac12 else x)
data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:4])

Diccionario ciudades disponibles

In [121]:
parroquia_dict = {
    '0101': 'Cuenca',
    '0901': 'Guayaquil',
    '0801': 'Esmeraldas',
    '0701': 'Machala',
    '1308': 'Manta',
    '1701': 'Quito',
    '1101': 'Loja',
    '1801': 'Ambato'
}

def get_parroquia(codigo):
    if codigo in parroquia_dict:
        return parroquia_dict[codigo]
    elif codigo[:2] in ['01', '02', '03', '04', '05', '06', '10', '11', '17', '18']:
        return 'Sierra'
    elif codigo[:2] in ['07', '08', '09', '12', '13', '23', '24']:
        return 'Costa'
    else:
        return 'Nacional'

data9['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)
data12['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)

In [123]:
data9['ciudad_asignada'].value_counts()

ciudad_asignada
Costa         6106
Guayaquil     4984
Sierra        4199
Nacional      3646
Cuenca        3040
Ambato        2905
Machala       2561
Quito         2268
Esmeraldas     330
Loja           214
Manta           64
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [124]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [125]:
data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [126]:
# Calculamos el deflactor
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])
data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

Ingreso promedio en el trimeste

In [127]:
data9['ingr_t3_r'] = data9['ingr_t3'] * data9['def_t3']
data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']

In [128]:
print(data9['ingr_t3_r'].mean())
print(data12['ingr_t4_r'].mean())

445.76671929935134
426.8790709740877


## Regiones

In [129]:
# Corregimos los códigos para usarlos cómo texto
data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == fac9-1 else x)

data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == fac12 else x)

data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

In [130]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [131]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data9['region'] = data9['ciudad_2'].map(codigo_region)

data12['region'] = data12['ciudad_2'].map(codigo_region)

In [133]:
data9['region'].value_counts()

region
Guayas                  6979
Sierra                  6477
Azuay                   3514
Amazonía                3226
El Oro                  3049
Pichincha               2682
Manabí                  1693
Costa, Santo Domingo    1421
Los Ríos                1098
Galápagos                178
Name: count, dtype: int64

In [134]:
data12['region'].value_counts()

region
Guayas                  6755
Sierra                  6559
Amazonía                3483
Azuay                   3426
El Oro                  3147
Pichincha               2772
Manabí                  1682
Costa, Santo Domingo    1525
Los Ríos                1134
Galápagos                163
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [135]:
data9['idef_hogar'] = data9['id_hogar']
data12['idef_hogar'] = data12['id_hogar']

print(len(data9['idef_hogar'].unique()))
print(len(data12['idef_hogar'].unique()))

8587
8756


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [136]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [137]:
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [138]:
print(data9['ingr_t3_h'].mean())
print(data12['ingr_t4_h'].mean())

639.1031926916079
617.2240895168709


## Sacamos edades negativas y mayores a 100 años

In [139]:
print(len(data9))
print(len(data12))

30317
30646


Transformamos las variables de edad a numericas para evitar problemas

In [140]:
data9['edad'] = pd.to_numeric(data9['p03'], errors='coerce')
data12['edad'] = pd.to_numeric(data12['p03'], errors='coerce')

In [141]:
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [142]:
k = 0.4
s = 0.9

In [143]:
# Si es necesario calcular el número de niños
data9['es_nino'] = data9['edad'] < 10
data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')

data12['es_nino'] = data12['edad'] < 10
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data9['es_adulto'] = data9['edad'] > 10
data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')

data12['es_adulto'] = data12['edad'] > 10
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [144]:
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [145]:
data9['ingr_t_t3'] = data9['ingr_t3_h'] / data9['escala']
data12['ingr_t_t4'] = data12['ingr_t4_h'] / data12['escala']

In [146]:
print(data9['ingr_t_t3'].mean())
print(data12['ingr_t_t4'].mean())

192.21412770690728
189.59336316013633


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [148]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2020

In [149]:
umbral_dict

{1: 78.919576103865,
 2: 78.9602207181246,
 3: 81.2706714423633,
 4: 81.1657430331955}

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [151]:
resultados_list = []

# Para cada trimeste
for t in [3, 4]:
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    salario = salario_dict.get(t)
    
    # Selecciona el dataframe correspondiente
    if t == 3:
        df_actual = data9
    elif t == 4:
        df_actual = data12
    else:
        continue

    # Agrupa por región
    grouped = df_actual.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2020,3,Amazonía,0.299906,0.179092,0.138681,0.114109,0.239278,0.413536,195.130500,136.696064,400.0,2.926200
1,2020,3,Azuay,0.177266,0.088429,0.063402,0.095775,0.190637,0.310663,219.799647,154.490648,400.0,2.589153
2,2020,3,"Costa, Santo Domingo",0.294837,0.149857,0.097886,0.075736,0.153531,0.244683,141.331053,109.613013,400.0,3.649202
3,2020,3,El Oro,0.279531,0.139941,0.105933,0.098824,0.203504,0.359998,169.398051,128.552795,400.0,3.111562
4,2020,3,Galápagos,0.152362,0.055784,0.025310,0.061801,0.127326,0.204306,315.097178,295.955134,400.0,1.351556
5,2020,3,Guayas,0.221983,0.074605,0.042068,0.068192,0.133642,0.204439,174.196910,132.369906,400.0,3.021835
6,2020,3,Los Ríos,0.249332,0.081872,0.037864,0.045445,0.088154,0.128833,134.040333,114.508779,400.0,3.493182
7,2020,3,Manabí,0.459092,0.158935,0.082562,0.063087,0.125791,0.198970,115.013154,86.345759,400.0,4.632538
8,2020,3,Pichincha,0.188845,0.106672,0.083563,0.115430,0.231985,0.381719,261.476883,166.457537,400.0,2.403015
9,2020,3,Sierra,0.291532,0.151262,0.106791,0.105986,0.215110,0.352456,194.352915,135.914269,400.0,2.943032


### Inserta los cálculos en la base final

In [152]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [153]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')